<a id="ksc2026-start"></a>
# KSC 2026 · GH200 × PhysicsNeMo 통합 실습

> **이 화면이 보이면 Jupyter 연결이 완료된 것입니다.** 브라우저는 참가자 컴퓨터에서 열리지만, Python과 GPU 계산은 Slurm이 동적으로 배정한 GH200 한 개에서 실행됩니다.

## 오늘의 실습 동선

| 시간 | 주제 | 바로 열기 | 확인할 결과 |
|---|---|---|---|
| 11:00–12:00 | GH200 소개 | 강의 슬라이드 | Grace CPU, Hopper GPU, CPU·GPU 일관성 메모리 구조 이해 |
| 13:30–14:00 | Grace CPU 컴파일·튜닝 | [01-1 노트북](01_GH200/01_CPU_Compile_and_Tune.ipynb) | OpenBLAS·NVPL 빌드와 성능 변화 |
| 14:00–14:30 | Hopper GPU 메모리·프로파일링 | [01-2 노트북](01_GH200/02_GPU_Memory_Profile.ipynb) | 메모리 접근 방식·Nsight Systems·nvbandwidth 비교 |
| 14:40–16:00 | PhysicsNeMo 기본/PINN | [02-1 노트북](02_PhysicsNeMo/01_Projectile_PINN.ipynb) | 초기조건과 운동방정식으로 발사체 궤적 근사 |
| 16:10–17:30 | 신경 연산자/FNO | [02-2 노트북](02_PhysicsNeMo/02_Poisson_FNO.ipynb) | 새로운 소스항에 대응하는 Poisson 해 예측 |
| 조기 완료 | FNO 푸리에 모드 비교 | [선택 노트북](02_PhysicsNeMo/optional/FNO_Mode_Ablation.ipynb) | 같은 조건에서 모드 수만 바꾼 결과 비교 |

- [전체 과정 안내](README.md)
- [01_GH200 상세 모듈 지도](01_GH200/README.md)
- [02_PhysicsNeMo 상세 모듈 지도](02_PhysicsNeMo/README.md)

## 먼저 알아둘 것

- **`ssh -N -L ...`을 실행한 로컬 터미널 탭은 실습 중 열어 둡니다.** 인증 뒤에 프롬프트나 메시지가 나타나지 않는 것이 정상입니다.
- 터널이 끊겼다면 PILOT 로그인 터미널에서 `/scratch/hackathon/ksc2026/bin/ksc2026`을 다시 실행하고, 새로 표시된 SSH 명령을 새 로컬 탭에서 실행합니다. 기존 Job이 살아 있으면 같은 세션으로 돌아갑니다.
- Jupyter는 60초마다 자동 저장합니다. 중요한 변경 뒤에는 macOS에서 `Cmd+S`, Windows·Linux에서 `Ctrl+S`를 누릅니다.
- 노트북과 결과 파일은 개인 `/scratch/<로그인계정>/ksc2026/workspaces/`에 저장됩니다. GitHub에 자동으로 올라가거나 다른 참가자의 파일과 섞이지 않습니다.
- 계산 노드에서는 `apt`, `pip`, `git`, `wget`, `curl`로 설치하거나 내려받지 않습니다. 문제가 생기면 셀의 전체 출력을 진행자에게 전달합니다.

## 폴더 안내

| 경로 | 용도 |
|---|---|
| `README.md` | 전체 과정과 핵심 개념 안내 |
| `01_GH200/README.md` | Grace CPU와 Hopper GPU 실습 지도 |
| `02_PhysicsNeMo/README.md` | PINN·FNO 실습 지도 |
| `labs/` | 노트북이 사용하는 소스 코드·설정·이미지 |

아래 환경 점검 셀을 위에서부터 실행합니다. 약 5분이 걸리며 데이터 생성이나 학습은 아직 시작하지 않습니다. 세 모듈이 모두 `READY`이면 첫 번째 GH200 노트북으로 이동합니다.


## 1. 과정 폴더 위치 확인

아래 셀은 현재 폴더에서 시작해 `00_Start_Here.ipynb`와 세 실습 지원 폴더가 함께 있는 과정 폴더를 찾습니다.


In [ ]:
from pathlib import Path

launch_dir = Path.cwd().resolve()

def is_course_root(path):
    return (
        (path / "00_Start_Here.ipynb").is_file()
        and (path / "labs" / "gh200").is_dir()
        and (path / "labs" / "projectile").is_dir()
        and (path / "labs" / "poisson_fno").is_dir()
    )

REPO_ROOT = next(
    (candidate for candidate in (launch_dir, *launch_dir.parents) if is_course_root(candidate)),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "KSC2026 과정 root를 찾지 못했습니다. 통합 image의 참가자 작업 폴더에서 "
        "00_Start_Here.ipynb를 열었는지 확인하세요."
    )

GH200_DIR = REPO_ROOT / "labs" / "gh200"
PROJECTILE_DIR = REPO_ROOT / "labs" / "projectile"
FNO_DIR = REPO_ROOT / "labs" / "poisson_fno"

print(f"Launch directory : {launch_dir}")
print(f"Course root      : {REPO_ROOT}")
print(f"GH200 lab        : {GH200_DIR}")
print(f"Projectile lab   : {PROJECTILE_DIR}")
print(f"Poisson FNO lab  : {FNO_DIR}")


## 2. 모듈별 필수 파일 확인

이 노트북 마지막의 링크를 순서대로 열면 됩니다. 필요한 경우 `01_GH200` 또는 `02_PhysicsNeMo` 폴더에서도 같은 노트북을 찾을 수 있습니다. `labs/`에는 노트북에서 사용하는 지원 코드가 들어 있습니다. 선택 실습은 `02_PhysicsNeMo/optional/`에 있으며, 현재는 FNO의 푸리에 모드 수를 비교하는 실습 하나만 포함합니다.


In [ ]:
session_files = {
    "01 GH200": [
        "01_GH200/01_CPU_Compile_and_Tune.ipynb",
        "01_GH200/02_GPU_Memory_Profile.ipynb",
        "labs/gh200/notebook_utils.py",
        "labs/gh200/blas/Makefile",
        "labs/gh200/blas/dgemm.c",
        "labs/gh200/cuda_memory/explicit.cu",
        "labs/gh200/cuda_memory/managed.cu",
        "labs/gh200/cuda_memory/hmm.cu",
    ],
    "02 PhysicsNeMo PINN": [
        "02_PhysicsNeMo/01_Projectile_PINN.ipynb",
        "labs/projectile/source_code/projectile.py",
        "labs/projectile/source_code/projectile_eqn.py",
        "labs/projectile/source_code/conf/config.yaml",
        "labs/projectile/images/projectile.svg",
        "labs/projectile/images/physicsnemo_sym_workflow.webp",
    ],
    "02 PhysicsNeMo FNO": [
        "02_PhysicsNeMo/02_Poisson_FNO.ipynb",
        "labs/poisson_fno/data_validation.py",
        "labs/poisson_fno/generate_data.py",
        "labs/poisson_fno/train_fno.py",
        "labs/poisson_fno/notebook_utils.py",
        "labs/poisson_fno/images/fno_data_flow.svg",
        "labs/poisson_fno/conf/config_FNO.yaml",
        "labs/poisson_fno/conf/config_FNO_recovery.yaml",
    ],
}

SESSION_FILES_READY = {}
for session, relative_paths in session_files.items():
    print(f"\n[{session}]")
    checks = []
    for relative in relative_paths:
        exists = (REPO_ROOT / relative).is_file()
        checks.append(exists)
        print(f"{'PASS' if exists else 'FAIL':4s}  {relative}")
    SESSION_FILES_READY[session] = all(checks)

optional_paths = [
    REPO_ROOT / "02_PhysicsNeMo" / "optional" / "README.md",
    REPO_ROOT / "02_PhysicsNeMo" / "optional" / "FNO_Mode_Ablation.ipynb",
]
print("\n[PhysicsNeMo optional]")
for optional_path in optional_paths:
    print(f"{'PASS' if optional_path.is_file() else 'WARN':4s}  {optional_path.relative_to(REPO_ROOT)}")


## 3. ARM64 개발 도구와 컨테이너 이미지 구성 확인

GH200 실습에는 `gcc`, `nvc`, `nvcc`, `nsys`, `nvbandwidth`, `make`가 모두 필요합니다. 컨테이너 이미지 구성 파일에는 PhysicsNeMo 25.11, NVHPC/NVPL 25.5, OpenBLAS 0.3.31, nvbandwidth 0.8이 기록되어 있어야 합니다. 아래에서는 **로컬 파일을 읽고 필요한 명령어가 설치돼 있는지만** 확인합니다.


In [ ]:
import json
import platform
import shutil

architecture = platform.machine().lower()
ARCH_READY = architecture in {"aarch64", "arm64"}
print(f"{'PASS' if ARCH_READY else 'FAIL'}  Architecture     {architecture} (expected aarch64/arm64)")

tool_names = ("gcc", "nvc", "nvcc", "nsys", "nvbandwidth", "make")
TOOL_PATHS = {name: shutil.which(name) for name in tool_names}
for name, path in TOOL_PATHS.items():
    print(f"{'PASS' if path else 'FAIL':4s}  {name:16s} {path or 'not found'}")
TOOLS_READY = all(TOOL_PATHS.values())

manifest_candidates = [
    Path("/etc/ksc2026-image.json"),
    REPO_ROOT / "container" / "ksc2026-image.json",
]
manifest_path = next((path for path in manifest_candidates if path.is_file()), None)
IMAGE_MANIFEST = None
if manifest_path is None:
    print("FAIL  Image manifest   /etc/ksc2026-image.json not found")
    MANIFEST_READY = False
else:
    IMAGE_MANIFEST = json.loads(manifest_path.read_text(encoding="utf-8"))
    manifest_text = json.dumps(IMAGE_MANIFEST, sort_keys=True)
    required_versions = ("25.11", "25.5", "0.3.31", "0.8")
    missing_versions = [value for value in required_versions if value not in manifest_text]
    MANIFEST_READY = not missing_versions
    print(f"{'PASS' if MANIFEST_READY else 'FAIL'}  Image manifest   {manifest_path}")
    if missing_versions:
        print(f"      Missing version markers: {missing_versions}")
    print(json.dumps(IMAGE_MANIFEST, indent=2, ensure_ascii=False))


## 4. Python, PhysicsNeMo, GPU 확인

### 4-1. `nvidia-smi`로 배정된 GPU 읽기

`nvidia-smi`는 NVIDIA 드라이버가 인식한 GPU 상태를 보여 주는 명령입니다. 이 노트북에서 실행하면 로그인 노드가 아니라 현재 Jupyter 커널이 실행 중인 **Slurm 계산 노드**의 GPU를 확인합니다.

Jupyter 코드 셀에서 Linux 명령을 실행할 때는 명령 앞에 `!`를 붙입니다. 따라서 아래 한 줄은 Python 코드가 아니라 계산 노드의 셸에 전달되는 명령입니다. 로그인 터미널에서는 느낌표 없이 `nvidia-smi`만 입력하면 됩니다.

- `GPU Name`: 계산 노드에서 보이는 GPU 모델
- `Driver Version`: 계산 노드에 설치된 NVIDIA 드라이버. 이 과정은 R570 이상을 요구하므로 `570.124.06`은 정상입니다.
- `Memory-Usage`: GPU 메모리의 현재 사용량과 총용량
- `GPU-Util`: 명령을 실행한 순간의 GPU 연산 사용률
- 표 상단의 `CUDA Version`: 드라이버가 지원하는 CUDA 호환 수준이며, 이미지 안의 `nvcc` 또는 PyTorch CUDA 버전과 반드시 같은 값은 아닙니다.

`nvidia-smi`만으로 CUDA 계산의 정상 동작까지 증명할 수는 없습니다. 다음 점검에서는 PyTorch로 각 GPU에 실제 텐서를 만들고 연산한 뒤 동기화합니다.


In [ ]:
!nvidia-smi


### 4-2. Python 패키지와 CUDA 연산 확인

모듈을 불러오지 못하면 패키지 이름과 오류를 함께 표시합니다. 이 과정의 모든 Slurm Job은 NVIDIA GH200 한 개를 요청하므로, PyTorch에서도 **GPU가 정확히 한 개 보여야** 환경 점검을 통과합니다. 계산 노드의 다른 GPU까지 보이거나 GPU가 전혀 보이지 않으면 정상적인 할당이 아닙니다.

`NVIDIA GH200 120GB`는 NVIDIA가 표시하는 장치 이름입니다. 이 시스템에서 `nvidia-smi`가 약 `97871 MiB`(약 `95.6 GiB`), PyTorch가 약 `95.0 GiB`를 표시하는 것은 정상입니다. 이 값은 CUDA에 보이는 GPU HBM 정보이며 현재 남은 여유 공간이나 Grace CPU 메모리가 아닙니다. 따라서 `GPU memory`는 정보만 표시하고 용량의 합격·불합격에는 사용하지 않습니다.


In [ ]:
import importlib
import shutil
import subprocess
import sys

modules_to_check = [
    ("torch", "PyTorch"),
    ("physicsnemo", "PhysicsNeMo"),
    ("physicsnemo.sym", "PhysicsNeMo-Sym"),
    ("h5py", "HDF5"),
    ("hydra", "Hydra"),
    ("matplotlib", "Matplotlib"),
]
imported = {}
import_errors = {}

print(f"Python           {sys.version.split()[0]}")
for module_name, display_name in modules_to_check:
    try:
        module = importlib.import_module(module_name)
        imported[module_name] = module
        version = getattr(module, "__version__", "version unavailable")
        print(f"PASS  {display_name:16s} {version}")
    except Exception as exc:
        import_errors[module_name] = f"{type(exc).__name__}: {exc}"
        print(f"FAIL  {display_name:16s} {import_errors[module_name]}")

STACK_READY = not import_errors
torch = imported.get("torch")
CUDA_READY = bool(torch is not None and torch.cuda.is_available())
GH200_READY = False
SM90_READY = False
GPU_COUNT_READY = False
CUDA_TENSOR_READY = False
DRIVER_READY = False
EXPECTED_GPU_COUNT = 1

visible_gpu_count = torch.cuda.device_count() if CUDA_READY else 0
GPU_COUNT_READY = visible_gpu_count == EXPECTED_GPU_COUNT
print(
    f"{'PASS' if GPU_COUNT_READY else 'FAIL'}  Visible GPUs     "
    f"{visible_gpu_count} (expected exactly 1)"
)

if CUDA_READY:
    gpu_name = torch.cuda.get_device_name(0)
    properties = torch.cuda.get_device_properties(0)
    GH200_READY = "GH200" in gpu_name.upper()
    SM90_READY = (properties.major, properties.minor) == (9, 0)
    print(f"PASS  CUDA             {torch.version.cuda}")
    print(f"{'PASS' if GH200_READY else 'FAIL'}  GPU              {gpu_name}")
    print(
        f"{'PASS' if SM90_READY else 'FAIL'}  Compute capability "
        f"{properties.major}.{properties.minor} (expected 9.0)"
    )
    print(f"INFO  GPU memory       {properties.total_memory / 2**30:.1f} GiB")
    try:
        for gpu_index in range(visible_gpu_count):
            with torch.cuda.device(gpu_index):
                probe = torch.arange(
                    1024, device=f"cuda:{gpu_index}", dtype=torch.float32
                )
                if float((probe * 2).sum().item()) != 1047552.0:
                    raise RuntimeError(f"GPU {gpu_index} tensor result mismatch")
                torch.cuda.synchronize(gpu_index)
        CUDA_TENSOR_READY = visible_gpu_count == EXPECTED_GPU_COUNT
        print(
            f"{'PASS' if CUDA_TENSOR_READY else 'FAIL'}  CUDA tensor op    "
            f"{visible_gpu_count} visible GPU(s) synchronized"
        )
    except Exception as exc:
        CUDA_TENSOR_READY = False
        print(f"FAIL  CUDA tensor op    {type(exc).__name__}: {exc}")
else:
    print("FAIL  CUDA GPU를 사용할 수 없습니다.")

if shutil.which("nvidia-smi"):
    query = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
        check=False,
    )
    detail = query.stdout.strip() or query.stderr.strip()
    NVIDIA_SMI_COMMAND_READY = query.returncode == 0
    print(f"{'PASS' if NVIDIA_SMI_COMMAND_READY else 'FAIL'}  nvidia-smi       {detail}")

    driver_query = subprocess.run(
        ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader,nounits"],
        capture_output=True,
        text=True,
        check=False,
    )
    versions = [line.strip() for line in driver_query.stdout.splitlines() if line.strip()]
    try:
        majors = [int(version.split(".", 1)[0]) for version in versions]
    except ValueError:
        majors = []
    DRIVER_READY = (
        driver_query.returncode == 0
        and len(majors) == visible_gpu_count
        and bool(majors)
        and all(major >= 570 for major in majors)
    )
    unique_versions = list(dict.fromkeys(versions))
    if len(unique_versions) == 1 and len(versions) > 1:
        driver_detail = f"{unique_versions[0]} ({len(versions)} visible GPUs)"
    else:
        driver_detail = ", ".join(unique_versions) if unique_versions else "unavailable"
    print(
        f"{'PASS' if DRIVER_READY else 'FAIL'}  NVIDIA driver    "
        f"{driver_detail} "
        "(R570 이상 필요; R570은 이미지의 CUDA forward compatibility 사용)"
    )
else:
    NVIDIA_SMI_COMMAND_READY = False
    print("FAIL  nvidia-smi       not found")


## 5. 참가자 작업 폴더 확인

학습 결과와 프로파일링 보고서는 지원 소스 코드 옆의 `work/` 또는 `outputs/` 폴더에 저장됩니다. 이 셀은 쓰기 권한을 확인하기 위해 작은 임시 파일을 만들었다가 즉시 삭제합니다.


In [ ]:
import tempfile

try:
    with tempfile.NamedTemporaryFile(prefix=".ksc_write_test_", dir=REPO_ROOT) as handle:
        handle.write(b"KSC2026 write check")
        handle.flush()
    WRITE_READY = True
    print("PASS  Participant workspace is writable")
except OSError as exc:
    WRITE_READY = False
    print(f"FAIL  Participant workspace is not writable: {exc}")

usage = shutil.disk_usage(REPO_ROOT)
print(f"INFO  Free disk space: {usage.free / 2**30:.1f} GiB")

COMMON_IMAGE_READY = (
    ARCH_READY
    and CUDA_READY
    and CUDA_TENSOR_READY
    and GH200_READY
    and SM90_READY
    and GPU_COUNT_READY
    and NVIDIA_SMI_COMMAND_READY
    and DRIVER_READY
    and MANIFEST_READY
)
READY_01 = SESSION_FILES_READY["01 GH200"] and COMMON_IMAGE_READY and TOOLS_READY and WRITE_READY
READY_02 = SESSION_FILES_READY["02 PhysicsNeMo PINN"] and COMMON_IMAGE_READY and STACK_READY and WRITE_READY
READY_03 = SESSION_FILES_READY["02 PhysicsNeMo FNO"] and COMMON_IMAGE_READY and STACK_READY and WRITE_READY

print("\n" + "=" * 72)
print(f"01 GH200 module               : {'READY' if READY_01 else 'CHECK REQUIRED'}")
print(f"02-1 PhysicsNeMo Projectile PINN: {'READY' if READY_02 else 'CHECK REQUIRED'}")
print(f"02-2 PhysicsNeMo Poisson FNO    : {'READY' if READY_03 else 'CHECK REQUIRED'}")
print("=" * 72)


## 결과 해석과 다음 단계

- 필수 파일 경로 옆에 `FAIL`이 보이면 올바른 참가자 작업 폴더를 열었는지 확인합니다.
- `Architecture`, 도구 이름 또는 `Image manifest` 옆에 `FAIL`이 보이면 현장에서 설치하지 말고 진행자에게 알립니다.
- `Visible GPUs`는 정확히 한 개일 때만 `PASS`입니다. 노트북 안에서는 Slurm이 배정한 물리 GPU가 `cuda:0`으로 보입니다.
- `NVIDIA driver 570.124.06`은 R570 계열입니다. 현재 검증된 SIF는 R570에서 CUDA 13 forward compatibility를 사용하며, 실제 CUDA 텐서 연산까지 통과해야 최종 정상으로 판정합니다.
- `CUDA`, `GPU`, `nvidia-smi` 또는 실제 텐서 연산 옆에 `FAIL`이 보이면 할당된 계산 노드와 GPU 연결 상태를 진행자에게 확인합니다.
- Python 모듈 이름 옆에 `FAIL`이 보이면 현재 Jupyter 커널이 KSC용 SIF의 커널인지 확인합니다.
- `Participant workspace is writable` 점검이 실패하면 개인 작업공간의 연결과 쓰기 권한을 진행자에게 확인합니다.

세 줄이 모두 `READY`라면 [01_GH200/01_CPU_Compile_and_Tune.ipynb](01_GH200/01_CPU_Compile_and_Tune.ipynb)로 이동합니다. 이후 [GPU 메모리와 프로파일링](01_GH200/02_GPU_Memory_Profile.ipynb), [발사체 운동 PINN](02_PhysicsNeMo/01_Projectile_PINN.ipynb), [Poisson FNO](02_PhysicsNeMo/02_Poisson_FNO.ipynb) 순서로 진행합니다.


---

## 출처와 라이선스

이 사전 점검 노트북은 KSC 2026 통합 과정용으로 작성했습니다. 저장소 각 파일의 기존 저작권과 라이선스 고지는 그대로 적용됩니다.
